# 10 · MINE-1 benchmark  (corpus-level, all graph variants)

**What this measures.** *Recall*: for each article we have 15 ground-truth fact sentences
(`dataset/metadata/<art>.json`). For every fact we query the extracted graph, retrieve a
sub-graph, verbalise it, and ask a judge whether that sub-graph **states the fact**.
Accuracy = fraction of the 15 facts that survive that test. This is the complement of nb9:
grounding asks *"is everything in the graph true to the article?"* (precision), MINE asks
*"is everything the article says actually in the graph?"* (recall). Neither number means much
without the other — report them as a pair.

**The original.** KGGen's MINE (Mo et al., 2025 — [arXiv:2502.09956](https://arxiv.org/abs/2502.09956),
`experiments/MINE/_1_evaluation.py`): embed the query, take the top-`k=8` nodes by cosine
similarity, walk 2 hops, join `"{subject} {predicate} {object}."` for every edge seen, and ask
GPT for a 1/0 on whether that text contains the answer. Their graphs are flat untyped string
triples, so that verbalisation is the whole graph.

**What changes here, and why.** Ours is typed and reified, so a literal port would score it
unfairly and would hide the things worth measuring:

| | MINE original | here |
|---|---|---|
| hop budget | 2 hops, flat triples | `HUB_FREE_HOP` — entering an n-ary hub is free, so `entity → hub → entity` costs the same 1 hop as a binary edge. Without this, reification looks worse purely because a fact's participants sit one node further apart. |
| verbalisation | `s p o.` | types folded in (`X (Country) location Y (Place).`) and hubs contribute their own statement — that typing *is* the output being evaluated |
| verdict | binary 1/0 | `found` / `partial` / `absent`, so "the graph has the event but dropped the death toll" is visible instead of being scored as a plain miss. Headline metric stays strict (`found` only) for comparability. |
| retrieval vs. coverage | conflated | optional **ceiling** run judges each fact against the *whole* graph. `ceiling − accuracy` is the share of facts the graph holds but retrieval failed to surface — an entirely different problem from a fact that was never extracted. |
| scope | one system | every variant in `GRAPH_VARIANTS` scored on the **same facts**, so e2e vs. staged, binary vs. reified, and pre- vs. post-grounding land in one comparable table. |

**Read the numbers with these caveats.** The judge sees only the retrieved context, so the score
is a property of *(extractor + retriever + verbaliser)*, not of the extractor alone — which is why
the ceiling column exists. And a benchmark of 15 facts per article resolves to ±6.7 points per
article, so per-article accuracy is noisy; the corpus mean is the number to quote, and it needs
~100 articles before its confidence interval is tight enough to separate close variants.
Validate the judge (§7) before trusting any of it.

In [1]:
# --- resolve Phase2_LLM_Extraction root (works whether this notebook lives at the
# project root or under notebooks/<track>/) so results/ & dataset/ paths are stable.
from pathlib import Path
def _find_root(start: Path = None):
    start = (start or Path.cwd()).resolve()
    for p in (start, *start.parents):
        if (p / 'results').is_dir() and (p / 'dataset').is_dir():
            return p
    return start
ROOT = _find_root()
# make Path writes create missing parent dirs; guarded so re-running the cell never double-wraps.
import pathlib as _pl
if not getattr(_pl.Path.write_text, '_mkparents', False):
    _wt, _wb = _pl.Path.write_text, _pl.Path.write_bytes
    def write_text(self, *a, **k):
        self.parent.mkdir(parents=True, exist_ok=True); return _wt(self, *a, **k)
    def write_bytes(self, *a, **k):
        self.parent.mkdir(parents=True, exist_ok=True); return _wb(self, *a, **k)
    write_text._mkparents = write_bytes._mkparents = True
    _pl.Path.write_text, _pl.Path.write_bytes = write_text, write_bytes
print("ROOT:", ROOT)

ROOT: /Users/marwah/Documents/Thesis/TriplesExtractionFromWebLLMs/Phase2_LLM_Extraction


In [3]:
# --- Config: MINE-1 benchmark --------------------------------------------------------------
import json, csv, re, time, random
from pathlib import Path
from collections import Counter, defaultdict, deque
import numpy as np

RESULTS_DIR = ROOT / "results"
META_DIR    = ROOT / "dataset" / "metadata"      # <art>.json -> {"title", "url", "facts": [15 sentences]}
EVAL_DIR    = RESULTS_DIR / "_global" / "evaluation"     # corpus-level reports land here

# --- what to score. (tag, track_dir, graph_filename). Every variant is scored on the SAME facts,
#     which is the whole point of the comparison table at the bottom. Drop lines you don't need —
#     each one costs a full pass of judge calls.
GRAPH_VARIANTS = [
    ("e2e_consolidated", "e2e",                           "nary_graph_dedup_extended_consolidated.json")
]

    # ("e2e_grounded",     "e2e",                           "nary_graph_grounded_fulltext.json"),
    # ("staged_nary",      "two_staged/3_nary_graph",       "nary_graph_extended.json"),
    # ("staged_binary",    "two_staged/2_binary_relations", "relations_consolidated.json"),

SELECTED_ARTICLES = None      # None -> every article with facts + that variant's graph; or ["art1", "art5"]

# --- retrieval (MINE's own settings: all-MiniLM-L6-v2, TOP_K=8, HOP_DEPTH=2) -----------------
EMBED_MODEL   = "intfloat/e5-base-v2"   # the pipeline's matcher; "all-MiniLM-L6-v2" for MINE parity
TOP_K         = 8            # seed nodes per fact
HOP_DEPTH     = 2            # hops walked out from each seed
HUB_FREE_HOP  = True         # entering an n-ary hub costs 0 hops -> reified and binary graphs get equal reach
INCLUDE_TYPES = True         # verbalise "Ebola outbreak (DiseaseOutbreak)" — the typing is part of the claim
NODE_ALIASES  = True         # fold entity aliases into the node's embedding document
INCLUDE_NODE_STATEMENTS = True   # a hub in context also contributes 'Event "trial begins" (ClinicalTrialEvent).'
MAX_CONTEXT_LINES = 150      # cap on retrieved context; ordered by seed rank, so the tail is what's cut

# How much retrieved context to keep ON DISK. In memory every row keeps all of it (the judge
# sample in section 7 needs it); this only trims the written log. "misses" keeps the context for
# facts that were not found — the rows you would actually open — and drops it for the rest, which
# is the difference between a ~10 KB and a ~220 KB file per article per variant.
STORE_CONTEXT = "misses"     # "misses" | "all" | "none"

GROUNDED_ONLY = True         # if the graph carries nb9 verdicts, score only grounded items
                             # (set False to measure what grounding costs in recall)

# --- judge -----------------------------------------------------------------------------------
GEMINI_MODEL = "gemini-2.5-flash"
JUDGE_BATCH  = 5             # facts per Gemini call
COUNT_PARTIAL_AS_FOUND = False   # headline stays strict/binary, as MINE is
RUN_CEILING  = True          # also judge each fact against the WHOLE graph -> coverage ceiling
CEILING_MAX_LINES = 600      # guard on the whole-graph context

print("variants:", [v[0] for v in GRAPH_VARIANTS])
print(f"retrieval: k={TOP_K} depth={HOP_DEPTH} hub_free={HUB_FREE_HOP} | judge: {GEMINI_MODEL} | ceiling: {RUN_CEILING}")

variants: ['e2e_consolidated']
retrieval: k=8 depth=2 hub_free=True | judge: gemini-2.5-flash | ceiling: True


In [4]:
# --- Gemini client -----------------------------------------------------------------------
%pip install -q google-genai

import os
from google import genai
from google.genai import types as genai_types

GEMINI_API_KEY = "" or os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
assert GEMINI_API_KEY, "Paste your key in GEMINI_API_KEY above, or set it in the env and restart the kernel."

client = genai.Client(api_key=GEMINI_API_KEY)

def gemini_json(system_prompt, user_message, model=None, temperature=0):
    """Call Gemini with JSON-mode output; return the parsed object."""
    resp = client.models.generate_content(
        model=model or GEMINI_MODEL,
        contents=user_message,
        config=genai_types.GenerateContentConfig(
            system_instruction=system_prompt,
            temperature=temperature,
            response_mime_type="application/json",
        ),
    )
    return json.loads(resp.text)

Note: you may need to restart the kernel to use updated packages.


In [5]:
# Embedding model (loaded once). Same convention as nb4/nb7: e5 wants "query: " on the query side
# and "passage: " on documents. Facts are queries, graph nodes are documents.
from sentence_transformers import SentenceTransformer
_MODEL = SentenceTransformer(EMBED_MODEL)

def _prefixes(name):
    n = name.lower()
    if "e5" in n:  return "query: ", "passage: "
    if "bge" in n: return "Represent this sentence for searching relevant passages: ", ""
    return "", ""
QP, DP = _prefixes(EMBED_MODEL)

def embed(texts, prefix):
    if not texts:
        return np.zeros((0, _MODEL.get_sentence_embedding_dimension()), dtype="float32")
    return np.asarray(_MODEL.encode([prefix + t for t in texts],
                                    normalize_embeddings=True, batch_size=64))
print("embedder ready:", EMBED_MODEL, "| query_prefix", repr(QP), "| doc_prefix", repr(DP))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

embedder ready: intfloat/e5-base-v2 | query_prefix 'query: ' | doc_prefix 'passage: '


## 1. Load the facts and the graph

The facts are the benchmark's ground truth: 15 sentences per article, each one a single
checkable claim from the source text. A graph is reduced to a `{id: node}` map plus an edge list,
generically enough that a **binary** graph (`relations_consolidated.json`, no `nary_nodes`) and a
**reified** one (`nary_graph_*.json`) both load — that's what makes the two directly comparable.

`GROUNDED_ONLY` drops the items nb9 rejected. An edge whose endpoint hub was dropped goes with it,
so the scored graph is exactly the graph a downstream consumer would see.

In [6]:
def article_ids_with_facts():
    ids = [p.stem for p in META_DIR.glob("*.json") if (json.loads(p.read_text()).get("facts"))]
    return sorted(ids, key=lambda a: (int(re.sub(r"\D", "", a) or 0), a))

def load_facts(article_id):
    p = META_DIR / f"{article_id}.json"
    return json.loads(p.read_text()).get("facts") if p.exists() else None

def variant_path(article_id, track_dir, filename):
    return RESULTS_DIR / article_id / track_dir / filename

def load_graph(article_id, track_dir, filename):
    return json.loads(variant_path(article_id, track_dir, filename).read_text())

def _grounded_ok(item):
    """True if the item survives nb9. Graphs without a grounding block pass unchanged."""
    if not GROUNDED_ONLY:
        return True
    g = item.get("grounding")
    return True if not g else bool(g.get("grounded", True))

def graph_nodes(graph):
    """id -> {kind, text, type, aliases}. Entities always; n-ary hubs when the graph has them."""
    nodes = {}
    for e in graph.get("entities", []):
        nodes[e["id"]] = {"kind": "entity", "text": e.get("text") or "",
                          "type": e.get("type"), "aliases": e.get("aliases") or []}
    for n in graph.get("nary_nodes", []):
        if not _grounded_ok(n):
            continue
        nodes[n["id"]] = {"kind": (n.get("node_kind") or "Event"), "text": n.get("label") or "",
                          "type": n.get("type"), "aliases": []}
    return nodes

def graph_edges(graph, nodes):
    """[{s, p, o}] — edges to a node dropped by GROUNDED_ONLY are dropped with it."""
    out = []
    for r in graph.get("relations", []):
        if not _grounded_ok(r):
            continue
        s, o = r.get("subject_id"), r.get("object_id")
        if s in nodes and o in nodes:
            out.append({"s": s, "p": r.get("property") or r.get("relation") or "related to", "o": o})
    return out

_ids = article_ids_with_facts()
print(f"{len(_ids)} articles carry facts:", _ids)
print("facts per article:", Counter(len(load_facts(a)) for a in _ids))
for tag, td, fn in GRAPH_VARIANTS:
    have = [a for a in _ids if variant_path(a, td, fn).exists()]
    print(f"  {tag:20} {len(have):3}/{len(_ids)} articles  ({td}/{fn})")

17 articles carry facts: ['art1', 'art4', 'art5', 'art6', 'art7', 'art8', 'art9', 'art10', 'art11', 'art12', 'art13', 'art14', 'art15', 'art16', 'art17', 'art18', 'art19']
facts per article: Counter({15: 17})
  e2e_consolidated       3/17 articles  (e2e/nary_graph_dedup_extended_consolidated.json)


## 2. Verbalising the graph

The judge cannot read `(EV1, location, E1)`, so every edge becomes a sentence. Types are carried
because they are part of what this pipeline asserts — a graph that gets the entity right and the
type wrong should not score full marks on a fact that names the type.

Hubs contribute their own statement (`Event "Ebola treatments trial begins" (ClinicalTrialEvent).`),
because in a reified graph the event's *existence and type* is a node, not an edge; without this a
fact like *"a trial has begun"* has nothing to match against. Set `INCLUDE_NODE_STATEMENTS = False`
for strict MINE parity (edges only) and expect reified variants to lose a few points.

In [7]:
def _label(nid, nodes):
    n = nodes.get(nid)
    if not n:
        return nid
    t = n.get("type")
    return f'{n["text"]} ({t})' if (INCLUDE_TYPES and t) else n["text"]

def verbalise_edge(e, nodes):
    return f'{_label(e["s"], nodes)} {e["p"]} {_label(e["o"], nodes)}.'

def node_statement(nid, nodes):
    """Only hubs get one: an entity's bare existence is not a claim worth showing the judge."""
    n = nodes.get(nid)
    if not n or n["kind"] == "entity":
        return None
    kind = str(n["kind"]).capitalize()
    return (f'{kind} "{n["text"]}" (type: {n["type"]}).' if (INCLUDE_TYPES and n.get("type"))
            else f'{kind} "{n["text"]}".')

def node_doc(nid, nodes):
    """What gets embedded for retrieval."""
    n = nodes[nid]
    doc = n["text"]
    if INCLUDE_TYPES and n.get("type"):
        doc += f' ({n["type"]})'
    if NODE_ALIASES and n.get("aliases"):
        doc += ". Also known as: " + ", ".join(n["aliases"])
    return doc

def verbalise_graph(nodes, edges, limit=None):
    """The whole graph as context — used for the coverage ceiling."""
    lines = [verbalise_edge(e, nodes) for e in edges]
    if INCLUDE_NODE_STATEMENTS:
        lines += [s for s in (node_statement(i, nodes) for i in nodes) if s]
    seen, out = set(), []
    for l in lines:
        if l not in seen:
            seen.add(l); out.append(l)
    return out[:limit] if limit else out

## 3. Retrieval — fact → sub-graph

MINE's procedure: cosine top-`k` nodes, then walk `HOP_DEPTH` hops collecting every edge crossed.

The one substantive change is `HUB_FREE_HOP`. In a binary graph the participants of a fact are
adjacent; in a reified one they hang off a hub, so `Congo → EV1 → WHO` is 2 hops where the binary
graph needs 1. Charging for that would measure the *encoding* rather than the *content*, so
entering a hub is free — the two encodings then get the same reach over the same fact. Turn it off
to see the effect; it is worth a sentence in the write-up either way.

In [8]:
def build_adjacency(edges):
    adj = defaultdict(list)
    for i, e in enumerate(edges):
        adj[e["s"]].append((e["o"], i))
        adj[e["o"]].append((e["s"], i))
    return adj

def expand(seed, nodes, adj, depth=None):
    """BFS from seed. Returns (edge indices crossed, nodes touched).
    With HUB_FREE_HOP, stepping INTO an n-ary hub costs 0 — leaving it costs the usual 1 — so
    entity -> hub -> entity totals the same 1 hop as the binary edge it stands for."""
    depth = HOP_DEPTH if depth is None else depth
    best = {seed: 0}
    q = deque([(seed, 0)])
    edge_ids, touched = [], {seed}
    seen_e = set()
    while q:
        node, d = q.popleft()
        for nbr, ei in adj.get(node, ()):
            step = 0 if (HUB_FREE_HOP and nodes.get(nbr, {}).get("kind", "entity") != "entity") else 1
            nd = d + step
            if nd > depth:
                continue
            if ei not in seen_e:
                seen_e.add(ei); edge_ids.append(ei)
            touched.add(nbr)
            if nd < best.get(nbr, 10**9):
                best[nbr] = nd
                q.append((nbr, nd))
    return edge_ids, touched

def retrieve(fact_vec, nodes, edges, adj, ids, emb):
    """Top-K seeds, then their neighbourhoods. Context keeps seed order, so the MAX_CONTEXT_LINES
    cut falls on the least relevant tail rather than at random."""
    if not ids:
        return [], []
    sims = emb @ fact_vec
    order = np.argsort(-sims)[:TOP_K]
    top = [(ids[i], round(float(sims[i]), 4)) for i in order]
    seen, lines = set(), []
    for nid, _ in top:
        eids, touched = expand(nid, nodes, adj)
        cand = [verbalise_edge(edges[i], nodes) for i in eids]
        if INCLUDE_NODE_STATEMENTS:
            cand += [s for s in (node_statement(t, nodes) for t in touched) if s]
        for l in cand:
            if l not in seen:
                seen.add(l); lines.append(l)
    return top, lines[:MAX_CONTEXT_LINES]

## 4. The judge

One call per `JUDGE_BATCH` facts. The judge sees the fact and the retrieved lines and nothing
else — no article, no outside knowledge — so it cannot credit the graph for something only the
world knows. `partial` exists because "the graph has the event but lost the death toll" and "the
graph never saw this" are different failures with different fixes; the headline metric still
counts `found` alone.

In [9]:
MINE_SYSTEM_PROMPT = """You are evaluating whether a knowledge graph contains a given fact.

For each item you are given:
- "fact": one sentence taken from the news article the graph was built from.
- "context": lines read off the knowledge graph. Each line is one graph assertion, verbalised.
  A line like 'X (Country) location Y (Place).' is an edge; a line like 'Event "..." (type: ...).'
  asserts that a node of that type exists.

Decide whether THE CONTEXT STATES THE FACT. Judge only against the context lines shown. Do not use
outside knowledge, and do not credit the graph for anything the context does not say, even if you
know it to be true in the world.

Verdicts:
- "found"   - the context states the fact, or states enough that the fact follows directly from it.
              Wording will differ; a paraphrase that preserves every element of the fact counts, and
              so does a fact spread across several context lines.
- "partial" - the context states some elements of the fact but is missing at least one of them:
              a quantity, a date, a place, or one of the named participants.
- "absent"  - the context does not state the fact.

Be strict about specifics. If the fact carries a number, a date, or a named participant and no
context line carries it, the verdict is at best "partial", never "found".

Return ONLY JSON:
{"results": [{"index": <int, copied from the item>,
              "verdict": "found" | "partial" | "absent",
              "support": [<at most 3 context lines, verbatim, that carry the fact>],
              "missing": "<short phrase naming what the context lacks, or null when found>"}]}
Return exactly one object per item given."""

def judge_facts(items, label=""):
    """items: [{"index", "fact", "context": [lines]}] -> {index: verdict dict}"""
    out = {}
    for b0 in range(0, len(items), JUDGE_BATCH):
        batch = items[b0:b0 + JUDGE_BATCH]
        payload = {"items": [{"index": it["index"], "fact": it["fact"],
                              "context": it["context"] or ["(the graph returned no context for this fact)"]}
                             for it in batch]}
        res = None
        for attempt in range(3):
            try:
                res = gemini_json(MINE_SYSTEM_PROMPT, json.dumps(payload, ensure_ascii=False))
                break
            except Exception as ex:
                if attempt == 2:
                    print(f"    judge failed {label} batch@{b0}: {type(ex).__name__}: {ex}")
                else:
                    time.sleep(2 * (attempt + 1))
        rows = res.get("results", []) if isinstance(res, dict) else (res or [])
        for r in rows:
            try:
                out[int(r["index"])] = r
            except (KeyError, TypeError, ValueError):
                continue
        for it in batch:      # a dropped item is a miss, not a hole in the table
            out.setdefault(it["index"], {"index": it["index"], "verdict": "absent",
                                         "support": [], "missing": "judge returned no verdict"})
    return out

def is_found(verdict):
    return verdict == "found" or (COUNT_PARTIAL_AS_FOUND and verdict == "partial")

## 5. Score one article

Two passes over the same facts: the **retrieved** context (the MINE number) and, when
`RUN_CEILING` is on, the **whole graph** (what the graph could answer with perfect retrieval).
`ceiling − accuracy` separates a retrieval problem from an extraction problem, which the original
benchmark leaves tangled.

In [10]:
def score_article(article_id, tag, track_dir, filename):
    facts = load_facts(article_id)
    graph = load_graph(article_id, track_dir, filename)
    nodes = graph_nodes(graph)
    edges = graph_edges(graph, nodes)
    adj   = build_adjacency(edges)

    ids = list(nodes)
    emb = embed([node_doc(i, nodes) for i in ids], DP)
    fvecs = embed(list(facts), QP)

    items, tops = [], {}
    for i, f in enumerate(facts):
        top, lines = retrieve(fvecs[i], nodes, edges, adj, ids, emb)
        tops[i] = top
        items.append({"index": i, "fact": f, "context": lines})
    judged = judge_facts(items, f"{article_id}/{tag}")

    ceiling = {}
    if RUN_CEILING:
        full = verbalise_graph(nodes, edges, CEILING_MAX_LINES)
        ceiling = judge_facts([{"index": i, "fact": f, "context": full} for i, f in enumerate(facts)],
                              f"{article_id}/{tag}/ceiling")

    rows = []
    for it in items:
        i = it["index"]
        j = judged.get(i, {})
        c = ceiling.get(i, {})
        rows.append({
            "article_id": article_id, "variant": tag, "fact_index": i, "fact": it["fact"],
            "verdict": j.get("verdict"), "found": is_found(j.get("verdict")),
            "missing": j.get("missing"),
            "ceiling_verdict": c.get("verdict") if RUN_CEILING else None,
            "ceiling_found": is_found(c.get("verdict")) if RUN_CEILING else None,
            "n_context_lines": len(it["context"]),
            "top_nodes": [f'{nodes[n]["text"]}@{s}' for n, s in tops[i]],
            "support": j.get("support") or [],
            "context": it["context"],
        })
    stats = {"n_nodes": len(nodes), "n_edges": len(edges),
             "n_hubs": sum(1 for n in nodes.values() if n["kind"] != "entity")}
    return rows, stats

# --- smoke test on one article ------------------------------------------------------------
_tag, _td, _fn = GRAPH_VARIANTS[0]
_aid = next(a for a in article_ids_with_facts() if variant_path(a, _td, _fn).exists())
_rows, _stats = score_article(_aid, _tag, _td, _fn)
print(f"{_aid} / {_tag}: {_stats}")
print(f"  accuracy {sum(r['found'] for r in _rows)}/{len(_rows)}"
      + (f"   ceiling {sum(bool(r['ceiling_found']) for r in _rows)}/{len(_rows)}" if RUN_CEILING else ""))
for r in _rows:
    mark = {"found": "OK  ", "partial": "PART", "absent": "MISS"}.get(r["verdict"], "??  ")
    print(f"  {mark} {r['fact'][:88]}")
    if r["verdict"] != "found":
        print(f"        missing: {r['missing']}   (ceiling: {r['ceiling_verdict']})")

art1 / e2e_consolidated: {'n_nodes': 129, 'n_edges': 156, 'n_hubs': 21}
  accuracy 12/15   ceiling 13/15
  OK   The Public Health France agency recorded 2,025 excess deaths between 22 June and 28 June
  OK   French health minister Stéphanie Rist reported an increase in deaths among individuals o
  OK   France experienced its highest country-wide average temperature on 24 June.
  PART Temperatures reached almost 41 degrees Celsius in Paris on 24 June.
        missing: the specific date for the temperature in Paris   (ceiling: partial)
  PART Half of France was placed under a red heat alert during the June heatwave.
        missing: the explicit temporal link between the red heat alert and the June heatwave   (ceiling: found)
  OK   The Copernicus climate service reported that Europe is heating up twice as fast as the g
  OK   Belgium recorded 1,222 excess deaths during the June heatwave.
  PART Almost half of the excess heatwave deaths in Belgium occurred in individuals aged 85 and
    

## 6. Batch — every variant, every article

One pass writes a per-article judgment log next to the graph it scored
(`mine_judgments_<tag>.json` / `.csv`, same shape as nb9's grounding log) and a corpus report to
`results/_global/evaluation/`.

Both a **macro** mean (average of per-article accuracies, ± std — what MINE reports) and a
**micro** rate (all facts pooled) are kept: macro is comparable to the paper, micro is the honest
corpus number when articles differ in difficulty. They diverge when a few articles fail badly, and
that divergence is itself worth reporting.

In [ ]:
JUDGMENT_COLS = ["article_id", "variant", "fact_index", "fact", "verdict", "found", "missing",
                 "ceiling_verdict", "ceiling_found", "n_context_lines", "top_nodes", "support"]

def _for_disk(r):
    keep = (STORE_CONTEXT == "all") or (STORE_CONTEXT == "misses" and not r["found"])
    return r if keep else {**r, "context": None}

def write_judgments(rows, article_id, tag, track_dir):
    d = RESULTS_DIR / article_id / track_dir
    (d / f"mine_judgments_{tag}.json").write_text(
        json.dumps([_for_disk(r) for r in rows], ensure_ascii=False, indent=2))
    with (d / f"mine_judgments_{tag}.csv").open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=JUDGMENT_COLS, extrasaction="ignore")
        w.writeheader()
        for r in rows:
            w.writerow({**r, "top_nodes": " | ".join(r["top_nodes"]), "support": " | ".join(r["support"])})

def summarise(rows_by_article, stats_by_article):
    per_article, accs = {}, []
    for aid, rows in rows_by_article.items():
        n = len(rows)
        acc = sum(r["found"] for r in rows) / n if n else 0.0
        cei = (sum(bool(r["ceiling_found"]) for r in rows) / n) if (RUN_CEILING and n) else None
        per_article[aid] = {
            "n_facts": n, "found": sum(r["found"] for r in rows),
            "accuracy": round(acc, 3),
            "partial": sum(r["verdict"] == "partial" for r in rows),
            "absent":  sum(r["verdict"] == "absent" for r in rows),
            "ceiling": round(cei, 3) if cei is not None else None,
            "retrieval_gap": round(cei - acc, 3) if cei is not None else None,
            **stats_by_article.get(aid, {}),
        }
        accs.append(acc)
    allr = [r for rows in rows_by_article.values() for r in rows]
    n = len(allr)
    micro = sum(r["found"] for r in allr) / n if n else 0.0
    micro_c = (sum(bool(r["ceiling_found"]) for r in allr) / n) if (RUN_CEILING and n) else None
    return {
        "n_articles": len(rows_by_article), "n_facts": n,
        "accuracy_micro": round(micro, 3),
        "accuracy_macro": round(float(np.mean(accs)), 3) if accs else None,
        "accuracy_std":   round(float(np.std(accs)), 3) if accs else None,
        "accuracy_median": round(float(np.median(accs)), 3) if accs else None,
        "ceiling_micro": round(micro_c, 3) if micro_c is not None else None,
        "retrieval_gap": round(micro_c - micro, 3) if micro_c is not None else None,
        "partial_rate": round(sum(r["verdict"] == "partial" for r in allr) / n, 3) if n else None,
        "per_article": per_article,
    }

def run_variant(tag, track_dir, filename, article_ids=None):
    ids = article_ids or article_ids_with_facts()
    ids = [a for a in ids if variant_path(a, track_dir, filename).exists()]
    rows_by_article, stats_by_article = {}, {}
    for aid in ids:
        try:
            rows, stats = score_article(aid, tag, track_dir, filename)
        except Exception as ex:
            print(f"  {aid:6} FAILED: {type(ex).__name__}: {ex}")
            continue
        write_judgments(rows, aid, tag, track_dir)
        rows_by_article[aid], stats_by_article[aid] = rows, stats
        acc = sum(r["found"] for r in rows)
        cei = sum(bool(r["ceiling_found"]) for r in rows) if RUN_CEILING else None
        print(f"  {aid:6} {acc:2}/{len(rows)}"
              + (f"   ceiling {cei:2}/{len(rows)}" if RUN_CEILING else "")
              + f"   ({stats['n_nodes']} nodes, {stats['n_edges']} edges)")
    return summarise(rows_by_article, stats_by_article), rows_by_article

REPORT, ALL_ROWS = {}, {}
_selected = SELECTED_ARTICLES or article_ids_with_facts()
for tag, td, fn in GRAPH_VARIANTS:
    print(f"\n=== {tag}  ({td}/{fn}) ===")
    rep, rows = run_variant(tag, td, fn, _selected)
    if rep["n_articles"]:
        REPORT[tag] = {"track_dir": td, "graph": fn, **rep}
        ALL_ROWS[tag] = rows
    else:
        print("  (no articles — skipped)")

In [ ]:
# --- the comparison table -------------------------------------------------------------------
CONFIG = {"embed_model": EMBED_MODEL, "top_k": TOP_K, "hop_depth": HOP_DEPTH,
          "hub_free_hop": HUB_FREE_HOP, "include_types": INCLUDE_TYPES,
          "include_node_statements": INCLUDE_NODE_STATEMENTS, "node_aliases": NODE_ALIASES,
          "max_context_lines": MAX_CONTEXT_LINES, "grounded_only": GROUNDED_ONLY,
          "store_context": STORE_CONTEXT,
          "judge_model": GEMINI_MODEL, "count_partial_as_found": COUNT_PARTIAL_AS_FOUND,
          "ran_ceiling": RUN_CEILING}

hdr = f'{"variant":22}{"arts":>5}{"facts":>7}{"micro":>8}{"macro":>8}{"std":>7}{"partial":>9}{"ceiling":>9}{"gap":>7}'
print(hdr); print("-" * len(hdr))
for tag, r in REPORT.items():
    print(f'{tag:22}{r["n_articles"]:5}{r["n_facts"]:7}'
          f'{r["accuracy_micro"]:8.3f}{r["accuracy_macro"]:8.3f}{r["accuracy_std"]:7.3f}'
          f'{r["partial_rate"]:9.3f}'
          f'{(r["ceiling_micro"] if r["ceiling_micro"] is not None else float("nan")):9.3f}'
          f'{(r["retrieval_gap"] if r["retrieval_gap"] is not None else float("nan")):7.3f}')

(EVAL_DIR / "mine_report.json").write_text(json.dumps({"config": CONFIG, "variants": REPORT},
                                                      ensure_ascii=False, indent=2))
with (EVAL_DIR / "mine_summary.csv").open("w", newline="", encoding="utf-8") as f:
    cols = ["variant", "track_dir", "graph", "n_articles", "n_facts", "accuracy_micro",
            "accuracy_macro", "accuracy_std", "accuracy_median", "partial_rate",
            "ceiling_micro", "retrieval_gap"]
    w = csv.DictWriter(f, fieldnames=cols, extrasaction="ignore")
    w.writeheader()
    for tag, r in REPORT.items():
        w.writerow({"variant": tag, **r})
print("\nwrote", (EVAL_DIR / "mine_report.json").relative_to(ROOT))
print("wrote", (EVAL_DIR / "mine_summary.csv").relative_to(ROOT))

In [ ]:
# --- where the misses are: which facts no variant recovers, and what they were missing -------
# Facts missed by EVERY variant are extraction failures, not encoding artefacts — that list is the
# one to read by hand, it is where the next prompt fix comes from.
if len(ALL_ROWS) >= 1:
    by_fact = defaultdict(dict)
    for tag, rows_by_article in ALL_ROWS.items():
        for aid, rows in rows_by_article.items():
            for r in rows:
                by_fact[(aid, r["fact_index"])][tag] = r
    universal = [(k, v) for k, v in by_fact.items()
                 if v and not any(r["found"] for r in v.values())]
    print(f"{len(universal)}/{len(by_fact)} facts missed by every variant scored\n")
    for (aid, fi), per_tag in universal[:20]:
        any_row = next(iter(per_tag.values()))
        print(f"  {aid} #{fi}: {any_row['fact'][:100]}")
        for tag, r in per_tag.items():
            print(f"      {tag:20} {r['verdict']:8} missing: {r['missing']}   ceiling={r['ceiling_verdict']}")
        print()

    print("\nmost common `missing` reasons across all misses:")
    for reason, n in Counter(r["missing"] for by_article in ALL_ROWS.values()
                             for rows in by_article.values() for r in rows
                             if not r["found"] and r["missing"]).most_common(15):
        print(f"  {n:4}  {reason}")

## 7. Validate the judge  ← do this before quoting any number above

Same discipline as nb9. The score is only as good as the judge producing it, and this judge is the
more error-prone of the two: it reads verbalised graph lines rather than natural prose, and the
`found` / `partial` boundary is exactly where a lenient judge inflates the benchmark.

Sample, label `y`/`n` by hand in the written file, re-run the scoring cell. Disagreements in the
lenient direction (judge said `found`, you said no) are the ones that change the headline.

In [ ]:
SAMPLE_N = 50
SAMPLE_CONTEXT_LINES = 40      # how much of the judge's context to reproduce for the human
random.seed(0)

def build_judge_sample(n=SAMPLE_N):
    pool = []
    for tag, rows_by_article in ALL_ROWS.items():
        for aid, rows in rows_by_article.items():
            for r in rows:
                pool.append({"article_id": aid, "variant": tag, "fact_index": r["fact_index"],
                             "fact": r["fact"], "support": r["support"],
                             "context": (r["context"] or [])[:SAMPLE_CONTEXT_LINES],
                             "judge": bool(r["found"])})
    # stratify so leniency and strictness are both testable
    yes = [p for p in pool if p["judge"]]
    no  = [p for p in pool if not p["judge"]]
    random.shuffle(yes); random.shuffle(no)
    half = n // 2
    sample = yes[:half] + no[:n - half]
    if len(sample) < n:
        sample += (yes[half:] + no[n - half:])[: n - len(sample)]
    random.shuffle(sample)
    for i, s in enumerate(sample):
        s["index"] = i; s["human"] = ""      # <- fill in "y" / "n"
    return sample

sample = build_judge_sample()
(EVAL_DIR / "mine_judge_sample.json").write_text(json.dumps(
    [{k: v for k, v in s.items() if k != "judge"} for s in sample], ensure_ascii=False, indent=2))
(EVAL_DIR / "mine_judge_key.json").write_text(json.dumps({str(s["index"]): s["judge"] for s in sample}, indent=2))
print("wrote", (EVAL_DIR / "mine_judge_sample.json").relative_to(ROOT),
      f"({len(sample)} items) — fill in the `human` field, then run the next cell")

In [ ]:
# Score human labels against the judge (run after filling in `human` in the sample file).
def score_agreement():
    labelled = json.loads((EVAL_DIR / "mine_judge_sample.json").read_text())
    key      = json.loads((EVAL_DIR / "mine_judge_key.json").read_text())
    pairs = [(str(x["index"]), (x.get("human") or "").strip().lower()) for x in labelled]
    pairs = [(i, h) for i, h in pairs if h in ("y", "n")]
    if not pairs:
        print("no human labels yet — fill the `human` field with y/n in mine_judge_sample.json")
        return
    agree = sum((h == "y") == bool(key[i]) for i, h in pairs)
    print(f"labelled {len(pairs)}/{len(labelled)} | agreement {agree}/{len(pairs)} = {agree/len(pairs):.1%}")
    lenient = sum(h == "n" and bool(key[i]) for i, h in pairs)      # judge found it, human did not
    strict  = sum(h == "y" and not bool(key[i]) for i, h in pairs)  # judge missed what human found
    print(f"  judge too lenient: {lenient}   judge too strict: {strict}")
    by_id = {str(x["index"]): x for x in labelled}
    for i, h in pairs:
        if (h == "y") != bool(key[i]):
            x = by_id[i]
            print(f"\n  [{i}] judge={bool(key[i])} human={h}  ({x['article_id']}/{x['variant']})")
            print(f"        fact: {x['fact'][:110]}")
            for s in (x["support"] or x["context"])[:2]:
                print(f"        ctx : {s[:110]}")

score_agreement()